# Tesla OTA Classifier - P2 : modeles finaux (heading+body)
Entraine SEULEMENT les 3 modeles heading+body (les gagnants, deja compares avant), les sauvegarde, et classifie les 2002 fonctions scrapees.

In [ ]:
!pip install transformers torch openpyxl pandas scikit-learn -q
print('Dependencies OK')

In [ ]:
from google.colab import files
print('Uploade CES 2 FICHIERS:')
print('1. p2_training_comparison_set.xlsx  (135 exemples avec labels)')
print('2. functions_final_p2.xlsx           (2002 fonctions a classifier)')
uploaded = files.upload()

In [ ]:
import pandas as pd
df = pd.read_excel('p2_training_comparison_set.xlsx', dtype={'branch': str})
all_functions = pd.read_excel('functions_final_p2.xlsx', dtype={'branch': str})
print(f"Training set: {len(df)} exemples")
print(f"A classifier: {len(all_functions)} fonctions")

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pickle

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class TeslaDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = list(texts); self.labels = list(labels)
        self.tokenizer = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                              padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze(),
                'label': torch.tensor(self.labels[idx], dtype=torch.long)}

def train_bert(texts, labels, num_labels, label_names, epochs=10, lr=2e-5, max_len=256):
    X_train, X_test, y_train, y_test = train_test_split(
        list(texts), list(labels), test_size=0.2, random_state=42, stratify=labels)
    train_ds = TeslaDataset(X_train, y_train, tokenizer, max_len)
    test_ds = TeslaDataset(X_test, y_test, tokenizer, max_len)
    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=8)

    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_labels).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            ids = batch['input_ids'].to(device); mask = batch['attention_mask'].to(device)
            lbl = batch['label'].to(device)
            out = model(ids, attention_mask=mask, labels=lbl)
            out.loss.backward(); optimizer.step()

    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in test_loader:
            ids = batch['input_ids'].to(device); mask = batch['attention_mask'].to(device)
            out = model(ids, attention_mask=mask)
            preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
            trues.extend(batch['label'].numpy())

    acc = accuracy_score(trues, preds)
    f1_macro = f1_score(trues, preds, average='macro')
    print(f'  Accuracy: {acc:.2%}  |  Macro-F1: {f1_macro:.2%}')
    print(classification_report(trues, preds, target_names=label_names))
    return model, acc, f1_macro

def predict(model, le, text, max_len=256):
    model.eval()
    enc = tokenizer(text, max_length=max_len, padding='max_length', truncation=True, return_tensors='pt')
    ids = enc['input_ids'].to(device); mask = enc['attention_mask'].to(device)
    with torch.no_grad():
        out = model(ids, attention_mask=mask)
        pred = torch.argmax(out.logits, dim=1).item()
    return le.inverse_transform([pred])[0]

print('Fonctions pretes.')

## Entrainement des 3 modeles finaux (heading+body uniquement)

In [ ]:
final_models = {}

tasks = [
    ('domain', 'domain_primary'),
    ('type', 'function_type'),
    ('coord', 'standalone_or_coord'),
]

for task_name, label_col in tasks:
    sub = df[df[label_col].notna()].copy()
    if task_name == 'type':
        sub = sub[sub[label_col] != 'REMOVED']
    le = LabelEncoder()
    sub['y'] = le.fit_transform(sub[label_col])

    print(f'=== {task_name} | n={len(sub)} | classes={list(le.classes_)} ===')
    model, acc, f1 = train_bert(sub['heading_body'], sub['y'], len(le.classes_),
                                 list(le.classes_), epochs=10, max_len=256)
    final_models[task_name] = {'model': model, 'label_encoder': le, 'accuracy': acc, 'f1_macro': f1}
    print()

## Sauvegarde des 3 modeles + encoders

In [ ]:
for task_name, m in final_models.items():
    torch.save(m['model'].state_dict(), f'{task_name}_headingbody_model.pt')
    with open(f'{task_name}_headingbody_encoder.pkl', 'wb') as f:
        pickle.dump(m['label_encoder'], f)
    files.download(f'{task_name}_headingbody_model.pt')
    files.download(f'{task_name}_headingbody_encoder.pkl')
    print(f'✅ {task_name}: {m["accuracy"]:.1%} accuracy | modele + encoder telecharges')

## Classification des 2002 fonctions

In [ ]:
all_functions['heading_body'] = all_functions['heading'].astype(str) + '. ' + all_functions['body'].astype(str)

for task_name, out_col in [('domain','domain_primary_pred'), ('type','function_type_pred'), ('coord','standalone_or_coord_pred')]:
    m = final_models[task_name]
    print(f'Classification: {task_name} ...')
    all_functions[out_col] = all_functions['heading_body'].apply(
        lambda t: predict(m['model'], m['label_encoder'], t)
    )

print('\nApercu:')
print(all_functions[['branch','heading','domain_primary_pred','function_type_pred','standalone_or_coord_pred']].head(10).to_string())

all_functions.to_excel('tesla_ota_classified_v3_headingbody.xlsx', index=False)
all_functions.to_csv('tesla_ota_classified_v3_headingbody.csv', index=False)
files.download('tesla_ota_classified_v3_headingbody.xlsx')
files.download('tesla_ota_classified_v3_headingbody.csv')
print('\n✅ Dataset final classifie (2002 fonctions) sauvegarde et telecharge.')